<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>》一书的配套代码，作者为 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 3 章：习题解答

本 Notebook 使用的 Python 包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.4
torch version: 2.7.1
tokenizers version: 0.21.4


&nbsp;
## 习题 3.1：添加更多测试用例

- 我们可以添加无穷多种不同的测试用例。
- 下面选取了一些有意思的例子。

In [ ]:
from reasoning_from_scratch.ch03 import (
    run_demos_table
)

more_tests = [
    # Different bracket types
    ("check_17", "[1, 2]", "(1, 2)", True),

    # Scientific notation
    ("check_18", "1e-3", "0.001", True),

    # Algebraic simplification with caret exponent
    ("check_19", "(-3)^2", "9", True),

    # Unicode minus (U+2212) vs ASCII hyphen-minus
    ("check_20", "−1", "-1", True),

]

run_demos_table(more_tests)

Test     | Expect | Got   | Status
check_17 | True   | True  | PASS  
check_18 | True   | True  | PASS  
check_19 | True   | True  | PASS  
check_20 | True   | False | FAIL  

Passed 3/4


- 可以看到，除了 `check_20` 之外，所有测试都通过了；`check_20` 把普通减号换成了肉眼几乎无法区分的 Unicode 版本减号。
- 可以在 `normalize_text` 函数的任意位置添加下面其中一行来修复这个测试用例：

```python
text = text.replace("−", "-")
# or
text = text.replace("\u2212", "-")
```

- 乍看之下，下面是另一个有意思的测试：

In [2]:
extra_tests_1 = [
    ("check_21", "Text around answer 3.", "3", True)
]

run_demos_table(extra_tests_1)

Test     | Expect | Got   | Status
check_21 | True   | False | FAIL  

Passed 0/1


- 这看起来可能意味着代码无法处理包含文本的答案，但实际上这是一个设计不佳的测试。
- 实际上，`run_demos_table` 函数的目的就是测试 `grade_answer` 函数，仅此而已。
- `grade_answer` 函数不会以这种形式接收完整答案，因为答案在传给它之前就已经从文本中提取出来了。

也就是说，如果要测试文本答案，需要按如下方式调用测试：

In [3]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate
)


extra_tests_2 = [
    ("check_21",
     extract_final_candidate("Text around answer 3."),
     "3", True)
]
run_demos_table(extra_tests_2)

Test     | Expect | Got  | Status
check_21 | True   | True | PASS  

Passed 1/1


&nbsp;
## 习题 3.2：计算平均回答长度

- 方案 A：可以修改 `evaluate_math500_stream` 函数，加入下面这些代码：

```python
# ...
# below `num_correct = 0`
total_len = 0

# ...
# inside for i, row in enumerate(math_data, start=1):
# anywhere below `gen_text = ...`
total_len += len(tokenizer.encode(gen_text))

# ...
# anywhere at the bottom before the return statement
avg_len = total_len / num_examples
print(f"Average length: {avg_len:.2f} tokens")
```

- 另一种方法是从运行主章中的 `evaluate_math500_stream` 函数时生成的 `.jsonl` 文件中计算回答长度。
- 首先按如下方式加载 `.jsonl` 文件：

In [5]:
import json
from pathlib import Path

WHICH_MODEL = "base"

dev_name = "mps"  # e.g., "cuda", "cpu"

# You may need to adjust this path:
local_path = Path(f"math500-{dev_name}.jsonl")
if not local_path.exists():
    raise FileNotFoundError(
        f"{local_path} not found. Run ch03_main.ipynb to create it."
    )

results = []
with open(local_path, "r") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

print("Number of entries:", len(results))


Number of entries: 10


- 注意，每条记录包含多个键，但我们只关心包含模型完整回答的 `"generated_text"` 键：

In [6]:
print(results[0].keys())

dict_keys(['index', 'problem', 'gtruth_answer', 'generated_text', 'extracted', 'correct'])


- 注意，每条记录包含多个键，但我们只关心包含模型完整回答的 `"generated_text"` 键：

In [7]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

✓ qwen3/tokenizer-base.json already up-to-date


- 然后可以按如下方式计算平均长度；这与修改 `evaluate_math500_stream` 函数的方式类似：

In [8]:
total_len = 0

for item in results:
    num_tokens = len(tokenizer.encode(item["generated_text"]))
    total_len += num_tokens

avg_len = total_len / len(results)
print(f"Average length: {avg_len:.2f} tokens")

Average length: 98.00 tokens


| 模式 | 设备 | 平均长度 | MATH-500 大小 |
|---|---|---:|---:|
| 基础 | CPU | 97.3 | 10 |
| 基础 | MPS | 98.0 | 10 |
| 推理 | CPU | 891.80 | 10 |
| 推理 | MPS | 1159.30 | 10 |
| | | | |
| 基础 | CUDA | 96.74 | 500 |
| 推理 | CUDA | 1361.21 | 500 |

- 正如预期，推理模型生成的回答长得多。

&nbsp;
## 习题 3.3：扩展或修改评估数据集

- 要在更大的数据集上评估模型，只需把 `math_data[:10]` 改为其他切片或更大的数量（最多 500）：

```python
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)
```

- 下表给出了不同数据集大小对应的准确率（由于 MATH-500 测试集已经打乱，因此没有额外进行打乱）。

| 模式 | 设备 | 准确率 | MATH-500 样例数 |
|---|---|---:|---:|
| 基础模型 | CUDA | 30.0% | 10 |
| 基础模型 | CUDA | 34.0% | 50 |
| 基础模型 | CUDA | 27.0% | 100 |
| 基础模型 | CUDA | 31.0% | 200 |
| 基础模型 | CUDA | 15.3% | 500 |
| | | | |
| 推理模型 | CUDA | 90.0% | 10 |
| 推理模型 | CUDA | 58.0% | 50 |
| 推理模型 | CUDA | 58.0% | 100 |
| 推理模型 | CUDA | 56.0% | 200 |
| 推理模型 | CUDA | 48.2% | 500 |

- 从上面的结果可以看到，前 10 个样例并不能很好地代表在全部 500 个样例上评估得到的 MATH-500 性能。

- 此外，我们也可以创建一个风格类似 MATH-500 的全新数据集。
- 例如，本仓库包含一个 MATH-500 风格的数据集；可以在主章中把文件名从 `math500_test.json` 改为 `math_new50_exercise.json` 来使用（该数据集位于本书 GitHub 仓库的 https://github.com/rasbt/reasoning-from-scratch/tree/main/ch03/01_main-chapter-code）。
- 基础模型和推理模型的性能如下：
    - 基础模型：36.0%（18/50）
    - 推理模型：80.0%（40/50）
- 由此可以得出，虽然原始 MATH-500 测试集可能已经包含在 Qwen3 的训练数据中，但模型在新的数学问题上表现相近，说明它并没有对原始 MATH-500 数据严重过拟合。

&nbsp;
## 习题 3.4：尝试不同的提示词模板

- 可以使用与本章建议类似的另一种提示词，把 `Question` 改为 `Problem`：

```python
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
```

- 使用该提示词后，基础模型在 500 个样例上的表现从 15.3% 提升到 31.2%。
- 由此可以得出，基础模型对提示词格式更加敏感（可能是因为训练数据中记忆了若干采用该提示词格式的 MATH-500 样例），而推理模型似乎基本不受影响。